In [1]:
from google.colab import drive
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from tqdm.auto import tqdm
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Young_Thug')

Mounted at /content/drive


In [2]:
# Create pipeline with extended max_length
pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    max_length=512,  # Use the model's maximum token limit
    truncation=True,
    padding=True
)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [3]:
def SentimentAnalyzer(df, columns, batch_size=16):
    """
    Perform sentiment analysis on specified columns of a DataFrame using batch processing.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): List of column names to analyze.
        label_mapping (dict): Mapping of model labels to human-readable labels.
        batch_size (int): Number of rows to process in each batch.

    Returns:
        pd.DataFrame: DataFrame with additional sentiment score and label columns.
    """
    for column in columns:
        if column not in df:
            raise ValueError(f"Column '{column}' not found in the DataFrame.")

        # Ensure all values are strings and handle NaN or None values
        df[column] = df[column].fillna("").astype(str)

        # Initialize empty lists to store results
        scores = []
        sentiments = []

        # Process data in batches
        tqdm.pandas(desc=f'Processing sentiment for {column}', colour='blue')
        for i in tqdm(range(0, len(df), batch_size), desc=f"Processing {column} in batches", colour="green"):
            # Get the batch of texts
            batch_texts = df[column][i:i+batch_size].tolist()

            # Run the pipeline on the batch
            results = pipe(batch_texts)

            # Extract scores and labels from results
            for res in results:
                scores.append(res['score'])
                sentiments.append(res['label'])

        # Add results to the DataFrame
        df[f'score_{column}'] = scores
        df[f'Sentiment_{column}'] = sentiments

    return df


## Newspaper

In [4]:
newspaper = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Sentiment Analysis/Newspaper/newspaper.csv', index_col=0)

In [5]:
newspaper = SentimentAnalyzer(newspaper, ['headline', 'snippet', 'article'])

Processing headline in batches:   0%|          | 0/96 [00:00<?, ?it/s]

Processing snippet in batches:   0%|          | 0/96 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/96 [00:00<?, ?it/s]

In [8]:
newspaper['Sentiment_headline'].value_counts()

,count
Sentiment_headline,
POSITIVE,824
NEGATIVE,701


In [9]:
newspaper.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Newspaper/newspaper.csv')

## Newstation

### WSB

In [13]:
wsb = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/News Station/wsb.csv', index_col=0)
wsb

,date,title,link,Keywords
0,2024-11-27,ysl trial jurors continue deliberating thanksg...,https://www.wsbtv.com/news/local/fulton-county...,ysl
1,2024-12-03,ysl trial jurors continue deliberating thanksg...,https://www.wsbtv.com/news/local/fulton-county...,ysl
2,2024-12-03,ysl trial jurors continue deliberating thanksg...,https://www.wsbtv.com/news/local/ysl-trial-jur...,ysl
3,2024-12-29,ysl trial jurors continue deliberating thanksg...,https://www.wsbtv.com/news/local/ysl-trial-jur...,ysl
4,2024-12-04,ysl trial yak gotti found guilty shannon still...,https://www.wsbtv.com/news/local/fulton-county...,"ysl, yak gotti, yak, gotti, shannon stillwell,..."
...,...,...,...,...
185,2024-10-08,cofounder says ysl music label believes crimes...,https://www.wsbtv.com/news/local/fulton-county...,ysl
186,2024-02-29,cofounder says ysl music label believes crimes...,https://www.wsbtv.com/news/local/fulton-county...,ysl
187,2024-06-10,cofounder says ysl music label believes crimes...,https://www.wsbtv.com/news/local/fulton-county...,ysl
188,2024-02-23,hundreds potential jurors questioned young thu...,https://www.wsbtv.com/news/local/fulton-county...,"young thug, ysl"


In [14]:
wsb = SentimentAnalyzer(wsb, ['title'])

Processing title in batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [23]:
wsb.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/News Station/wsb.csv')

### GPB

In [20]:
gpb = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/News Station/gpb.csv', index_col = 0)
gpb = SentimentAnalyzer(gpb, ['headlines', 'snippets', 'article'])
gpb

Processing headlines in batches:   0%|          | 0/4 [00:00<?, ?it/s]

Processing snippets in batches:   0%|          | 0/4 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/4 [00:00<?, ?it/s]

,publication_date,headlines,snippets,article,links,Keywords,score_headlines,Sentiment_headlines,score_snippets,Sentiment_snippets,score_article,Sentiment_article
1,2024-07-17,judge young thugs trial recused case,fulton county chief judge ural glanville taken...,caption young thug performs halftime game atla...,https://www.gpb.org/news/2024/07/17/why-the-ju...,"young thug, ysl, ysl rico, rico, jeffrey willi...",0.524498,NEGATIVE,0.996927,NEGATIVE,0.962516,NEGATIVE
2,2024-06-11,defense attorney rapper young thug found conte...,georgia judge ordered lawyer rapper young thug...,,https://www.gpb.org/news/2024/06/11/defense-at...,young thug,0.993810,NEGATIVE,0.987871,NEGATIVE,0.748121,POSITIVE
3,2024-12-03,defendants guilty murder gang trial led rapper...,longrunning gang racketeering trial atlanta le...,,https://www.gpb.org/news/2024/12/03/last-2-def...,young thug,0.938057,NEGATIVE,0.993061,NEGATIVE,0.748121,POSITIVE
4,2024-11-01,georgia today day early voting dock collapse w...,friday nov,,https://www.gpb.org/news/2024/11/01/georgia-to...,young thug,0.987573,NEGATIVE,0.989105,POSITIVE,0.748121,POSITIVE
5,2023-11-27,georgia today service rosalynn carter underway...,monday nov,,https://www.gpb.org/news/2023/11/27/georgia-to...,young thug,0.750480,POSITIVE,0.975707,POSITIVE,0.748121,POSITIVE
6,2023-11-09,georgia today actors strike ends young thug ly...,thursday nov edition georgia today actors stri...,,https://www.gpb.org/news/2023/11/09/georgia-to...,young thug,0.991484,NEGATIVE,0.657480,NEGATIVE,0.748121,POSITIVE
7,2022-05-11,rapper gunna booked jail racketeering charge,rapper gunna booked jail atlanta wednesday rac...,,https://www.gpb.org/news/2022/05/11/rapper-gun...,"young thug, gunna",0.988217,NEGATIVE,0.983894,NEGATIVE,0.748121,POSITIVE
9,2022-12-15,rapper gunna pleads guilty racketeering case a...,rapper gunna pleaded guilty atlanta racketeeri...,,https://www.gpb.org/news/2022/12/15/rapper-gun...,"young thug, gunna",0.935927,NEGATIVE,0.951874,NEGATIVE,0.748121,POSITIVE
10,2022-07-07,rapper gunna denied bond gang racketeering case,judge atlanta denied bond rapper gunna charged...,,https://www.gpb.org/news/2022/07/07/rapper-gun...,"young thug, gunna",0.747962,POSITIVE,0.924698,NEGATIVE,0.748121,POSITIVE
11,2024-09-05,rich homie quan atlanta rapper known trap jams...,rich homie quan died atlanta rapper gained mai...,,https://www.gpb.org/news/2024/09/05/rich-homie...,young thug,0.985223,NEGATIVE,0.720304,NEGATIVE,0.748121,POSITIVE


In [22]:
gpb.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/News Station/gpb.csv')

### FOX 5

In [25]:
fox5 = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/News Station/fox5.csv', index_col=0)
fox5 = SentimentAnalyzer(fox5, ['headlines', 'snippets', 'article'])
fox5

Processing headlines in batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processing snippets in batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/32 [00:00<?, ?it/s]

,publication_date,headlines,snippets,article,links,Keywords,score_headlines,Sentiment_headlines,score_snippets,Sentiment_snippets,score_article,Sentiment_article
1,2024-12-17,ysl rico case rapper yak gotti remain jail sha...,codefendants ysl rico trial appeared court tue...,codefendants ysl rico trial appear court final...,https://www.fox5atlanta.com/news/ysl-case-rapp...,"ysl, ysl rico, rico, yak gotti, yak, gotti, sh...",0.973808,NEGATIVE,0.970166,NEGATIVE,0.990262,NEGATIVE
2,2024-12-17,codefendants ysl rico trial appear court,defendants ysl rico trial leading classaction ...,,https://www.fox5atlanta.com/video/1563659,"ysl, ysl rico, rico",0.863423,NEGATIVE,0.984963,NEGATIVE,0.748121,POSITIVE
3,2024-12-17,deamonte kendrick shannon stillwell court,little month rapper young thug pleaded guilty ...,,https://www.fox5atlanta.com/video/1563487,"shannon stillwell, shannon, stillwell, deamont...",0.856045,POSITIVE,0.980961,NEGATIVE,0.748121,POSITIVE
4,2024-12-12,ysl rico trial defendant files classaction law...,suwanee man exploited people recovering drug a...,article fulton county jail fox brief classacti...,https://www.fox5atlanta.com/news/ysl-rico-tria...,"ysl, ysl rico, rico",0.989519,NEGATIVE,0.990594,NEGATIVE,0.998093,NEGATIVE
5,2024-12-11,young thug allowed visit atlanta home starting,attorneys deamonte kendrick goes stage yak got...,young thug gets atlanta ban terms amended judg...,https://www.fox5atlanta.com/news/young-thug-al...,young thug,0.947914,POSITIVE,0.933875,NEGATIVE,0.992413,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...,...,...
1287,2018-05-07,puerto rico mourns loss airmen killed savannah...,men came island joined mission retire oldest p...,image lt david albandoz senior airman roberto ...,https://www.fox5atlanta.com/news/puerto-rico-m...,rico,0.994243,NEGATIVE,0.552442,POSITIVE,0.985099,NEGATIVE
1293,2018-04-18,excavator blamed islandwide blackout puerto rico,islandwide blackout hit puerto rico wednesday ...,article damian entwistle flickr san juan puert...,https://www.fox5atlanta.com/news/excavator-bla...,rico,0.981689,NEGATIVE,0.997070,NEGATIVE,0.995138,NEGATIVE
1301,2018-03-05,huge waves slam puerto rico force evacuations,waves nearly feet high us winter storm slammed...,article photo courtesy us coast guard san juan...,https://www.fox5atlanta.com/news/huge-waves-sl...,rico,0.693429,POSITIVE,0.979474,NEGATIVE,0.997271,NEGATIVE
1307,2018-01-29,southwest airlines flies dogs cats puerto rico...,fox news talk lucky dogs,image shelley castle photography lucky dog res...,https://www.fox5atlanta.com/news/southwest-air...,rico,0.949462,NEGATIVE,0.945697,POSITIVE,0.962979,NEGATIVE


In [29]:
fox5.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/News Station/fox5.csv')

#### FOX5 Youtube

In [5]:
fox5_youtube = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/News Station/fox5 YouTube.csv', low_memory = False)
fox5_youtube = SentimentAnalyzer(fox5_youtube, ['title', 'comment'], batch_size = 120)
fox5_youtube

Processing title in batches:   0%|          | 0/631 [00:00<?, ?it/s]

Processing comment in batches:   0%|          | 0/631 [00:00<?, ?it/s]

,Unnamed: 0,video_id,upload_date,title,views,video likes,total comment,published_at,comment,comment likes,replies,Keywords,score_title,Sentiment_title,score_comment,Sentiment_comment
0,0,rRPiY8MPcG4,2024-12-17 23:05:49+00:00,codefendants ysl rico trial appear court fox news,749,3,1,2024-12-19 10:07:44+00:00,plea deals murderers insane,0,0,"ysl, ysl rico, rico",0.923903,NEGATIVE,0.981367,NEGATIVE
1,1,VCdbAmV6Oa0,2024-12-17 20:41:46+00:00,deamonte kendrick shannon stillwell court fox ...,428,7,2,2024-12-17 22:45:17+00:00,nice pink jacket work tv nice choicewomanfacep...,0,0,"shannon stillwell, shannon, stillwell, deamont...",0.862319,POSITIVE,0.992314,POSITIVE
2,2,VCdbAmV6Oa0,2024-12-17 20:41:46+00:00,deamonte kendrick shannon stillwell court fox ...,428,7,2,2024-12-17 22:33:59+00:00,let men home,0,0,"shannon stillwell, shannon, stillwell, deamont...",0.862319,POSITIVE,0.694454,POSITIVE
3,3,XEEohTqbwUE,2024-12-12 20:35:13+00:00,watch live amari hall murder celeste owens goe...,14004,198,13,2024-12-17 21:29:13+00:00,cryingface,0,0,NaN,0.926170,NEGATIVE,0.990796,NEGATIVE
4,4,XEEohTqbwUE,2024-12-12 20:35:13+00:00,watch live amari hall murder celeste owens goe...,14004,198,13,2024-12-17 17:12:00+00:00,precious little girl given mom able stay grand...,2,0,NaN,0.926170,NEGATIVE,0.593628,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75605,75605,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,watch live hearings ysl defendants,8711,101,8,2024-12-17 19:56:12+00:00,went trial,0,0,ysl,0.991000,POSITIVE,0.983897,POSITIVE
75606,75606,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,watch live hearings ysl defendants,8711,101,8,2024-12-17 19:27:27+00:00,thankyou fox bond hearings fo yak amp shannon ...,0,0,"ysl, yak, shannon",0.991000,POSITIVE,0.978709,POSITIVE
75607,75607,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,watch live hearings ysl defendants,8711,101,8,2024-12-17 19:25:55+00:00,wish wellamp prosecutors ashamed jury found gu...,2,0,ysl,0.991000,POSITIVE,0.994674,NEGATIVE
75608,75608,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,watch live hearings ysl defendants,8711,101,8,2024-12-17 19:07:01+00:00,know trial,0,0,ysl,0.991000,POSITIVE,0.967984,POSITIVE


In [8]:
fox5_youtube['Sentiment_title'].value_counts()

,count
Sentiment_title,
NEGATIVE,61529
POSITIVE,14081


In [9]:
fox5_youtube.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/News Station/fox5 YouTube.csv')

## Magazine

### People

In [13]:
people = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/People.csv', index_col=0)
people = SentimentAnalyzer(people, ['headlines', 'article'])
people

Processing headlines in batches:   0%|          | 0/15 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/15 [00:00<?, ?it/s]

,publication_date,headlines,article,links,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
1,2024-07-15,know highprofile young thug rico case,young thug photo prince williamswireimage high...,https://people.com/young-thug-ysl-rico-trail-w...,"young thug, gunna, ysl, ysl rico, rico, jeffre...",0.944002,NEGATIVE,0.987650,NEGATIVE
2,2024-07-15,know highprofile young thug rico case,highprofile trial involving rapper young thug ...,https://people.com/young-thug-ysl-rico-trail-w...,"young thug, gunna, ysl, ysl rico, rico, jeffre...",0.944002,NEGATIVE,0.986918,NEGATIVE
3,2024-11-01,young thugs girlfriend mariah scientist relati...,young thug speaks onstage revolt summit novemb...,https://people.com/who-is-mariah-the-scientist...,"young thug, rico, sb, williams",0.861573,NEGATIVE,0.988755,NEGATIVE
4,2024-11-01,young thugs girlfriend mariah scientist relati...,young thug special duet partner life girlfrien...,https://people.com/who-is-mariah-the-scientist...,"young thug, rico, sb, williams",0.861573,NEGATIVE,0.993752,NEGATIVE
5,2024-10-31,young thug pleads guilty criminal activity cha...,young thug photo paras griffingetty young thug...,https://people.com/young-thug-pleads-guilty-in...,"young thug, gunna, ysl, rico, sb, williams, ro...",0.935545,NEGATIVE,0.991841,NEGATIVE
...,...,...,...,...,...,...,...,...,...
236,2017-07-13,new england patriots danny amendola confirms h...,people probably familiar danny amendola reason...,https://people.com/style/danny-amendola-signs-...,young thug,0.995704,POSITIVE,0.779582,POSITIVE
237,2022-01-06,snl sets ariana debose host roddy ricch musica...,ariana debose roddy ricch photo getty saturday...,https://people.com/tv/snl-sets-first-2022-show...,young thug,0.987183,POSITIVE,0.986662,NEGATIVE
238,2022-01-06,snl sets ariana debose host roddy ricch musica...,saturday night live thursday nbc sketch comedy...,https://people.com/tv/snl-sets-first-2022-show...,young thug,0.987183,POSITIVE,0.986561,NEGATIVE
239,2020-10-27,bet hip hop awards know starstudded event,roddy ricch megan thee stallion drake photo ge...,https://people.com/music/bet-hip-hop-awards-20...,"young thug, gunna",0.965520,POSITIVE,0.844183,NEGATIVE


In [15]:
os.makedirs('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/', exist_ok= True)
people.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/People.csv')

### Shade Room

In [17]:
shade_room = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/Shade Room.csv', index_col=0)
shade_room = SentimentAnalyzer(shade_room, ['headlines', 'article'])
shade_room

Processing headlines in batches:   0%|          | 0/6 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/6 [00:00<?, ?it/s]

,publication_date,headlines,article,links,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
0,2024-11-23,cmon princess treatment mariah scientist spark...,man man man person definitely mariah scientist...,https://theshaderoom.com/mariah-the-scientist-...,"young thug, ysl, ysl rico, rico, paige reese w...",0.947653,NEGATIVE,0.987669,NEGATIVE
1,2024-11-10,whew gunnas brother weighs young thugs x accou...,whew young thugs x account appeared post delet...,https://theshaderoom.com/gunnas-brother-respon...,"young thug, gunna, ysl",0.989226,NEGATIVE,0.998771,NEGATIVE
2,2024-11-09,social media goes reactions young thugs x acco...,whew chile innanet going young thugs x account...,https://theshaderoom.com/young-thug-gunna-mess...,"young thug, gunna, ysl, ysl rico, rico",0.995937,NEGATIVE,0.998358,NEGATIVE
3,2024-11-04,whew social media users upset mariah scientist...,social media users upset mariah scientist inte...,https://theshaderoom.com/social-media-users-re...,young thug,0.998230,NEGATIVE,0.996410,NEGATIVE
4,2024-11-04,young thug drops messages social media release...,young thug shared message x twitter release ja...,https://theshaderoom.com/young-thug-message-so...,"young thug, ysl, ysl rico, rico",0.998560,NEGATIVE,0.994807,NEGATIVE
...,...,...,...,...,...,...,...,...,...
81,2022-03-18,lakevia jackson mother young thugs son fatally...,scary time casual spaces meant joy nt safe mot...,https://theshaderoom.com/lakevia-jackson-mothe...,"young thug, rico, sb, williams",0.985818,NEGATIVE,0.988803,NEGATIVE
82,2020-12-28,young thug seemingly responds jerrika karlaes ...,nt hip hops longstanding couples young thug je...,https://theshaderoom.com/young-thug-seemingly-...,young thug,0.951352,NEGATIVE,0.998093,NEGATIVE
83,2015-03-25,roommate talk dear tsr young thug stole yeezys...,roommate talk posts percent user submitted ref...,https://theshaderoom.com/roommate-talk-dear-ts...,young thug,0.989008,POSITIVE,0.996645,NEGATIVE
84,NaN,roommate talk dear tsr ti young thug showed bb...,,https://theshaderoom.com/roommate-talk-dear-ts...,young thug,0.987061,NEGATIVE,0.748121,POSITIVE


In [18]:
shade_room.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/Shade Room.csv')

### Rolling Stone

In [20]:
rollingstone = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/rollingstone.csv', index_col =0)
rollingstone = SentimentAnalyzer(rollingstone, ['headlines', 'article'])
rollingstone.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/rollingstone.csv')
rollingstone

Processing headlines in batches:   0%|          | 0/65 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/65 [00:00<?, ?it/s]

,publication_date,headlines,article,links,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
0,2024-12-23,watch tyla join gunna jump blockbuster nigeria...,stage lagos nigeria time gunna surprised fans ...,https://www.rollingstone.com/music/music-news/...,"young thug, gunna, sergio kitchens, sergio, ki...",0.997775,POSITIVE,0.977883,NEGATIVE
1,2024-12-23,watch tyla join gunna jump blockbuster nigeria...,stage lagos nigeria time gunna surprised fans ...,https://www.rollingstone.com/music/music-news/...,"young thug, gunna, sergio kitchens, sergio, ki...",0.997775,POSITIVE,0.977883,NEGATIVE
2,2024-12-23,watch tyla join gunna jump blockbuster nigeria...,stage lagos nigeria time gunna surprised fans ...,https://www.rollingstone.com/music/music-news/...,"young thug, gunna, sergio kitchens, sergio, ki...",0.997775,POSITIVE,0.977883,NEGATIVE
3,2024-12-04,happened drakekendrick beef,kendrick lamar took aim drake verse future met...,https://www.rollingstone.com/music/music-news/...,"young thug, rico, williams, kendrick",0.813216,NEGATIVE,0.992684,NEGATIVE
4,2024-12-04,happened drakekendrick beef,kendrick lamar took aim drake verse future met...,https://www.rollingstone.com/music/music-news/...,"young thug, rico, williams, kendrick",0.813216,NEGATIVE,0.992684,NEGATIVE
...,...,...,...,...,...,...,...,...,...
1155,2018-08-02,stakes travis scott higher astroworld potentia...,ask casual fan like travis scott likely hear t...,https://www.rollingstone.com/music/music-news/...,young thug,0.998875,POSITIVE,0.991705,NEGATIVE
1156,2018-07-16,vmas cardi b carters childish gambino drake le...,cardi b carters beyonc jayz childish gambino d...,https://www.rollingstone.com/music/music-news/...,"young thug, sb, kendrick",0.577043,NEGATIVE,0.837003,POSITIVE
1157,2018-07-16,vmas cardi b carters childish gambino drake le...,cardi b carters beyonc jayz childish gambino d...,https://www.rollingstone.com/music/music-news/...,"young thug, sb, kendrick",0.577043,NEGATIVE,0.837003,POSITIVE
1158,2018-07-12,taylor bennett getting comfortable,streak fierce independence taylor bennett happ...,https://www.rollingstone.com/music/music-news/...,young thug,0.999595,POSITIVE,0.851910,NEGATIVE


### Time

In [23]:
times = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/time.csv', index_col=0)
times = SentimentAnalyzer(times, ['headlines', 'article'])
times.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/Times.csv')
times

Processing headlines in batches:   0%|          | 0/6 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/6 [00:00<?, ?it/s]

,links,publication_date,article,headlines,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
1,https://theshaderoom.com/mariah-the-scientist-...,2024-11-23,man man man person definitely mariah scientist...,,"young thug, ysl, ysl rico, rico, paige reese w...",0.748121,POSITIVE,0.987669,NEGATIVE
2,https://theshaderoom.com/gunnas-brother-respon...,2024-11-10,whew young thugs x account appeared post delet...,,"young thug, gunna, ysl",0.748121,POSITIVE,0.998771,NEGATIVE
3,https://theshaderoom.com/young-thug-gunna-mess...,2024-11-09,whew chile innanet going young thugs x account...,,"young thug, gunna, ysl, ysl rico, rico",0.748121,POSITIVE,0.998358,NEGATIVE
4,https://theshaderoom.com/social-media-users-re...,2024-11-04,social media users upset mariah scientist inte...,,young thug,0.748121,POSITIVE,0.996410,NEGATIVE
5,https://theshaderoom.com/young-thug-message-so...,2024-11-04,young thug shared message x twitter release ja...,,"young thug, ysl, ysl rico, rico",0.748121,POSITIVE,0.994807,NEGATIVE
...,...,...,...,...,...,...,...,...,...
83,https://theshaderoom.com/roommate-talk-dear-ts...,2015-03-25,roommate talk posts percent user submitted ref...,,young thug,0.748121,POSITIVE,0.996645,NEGATIVE
84,https://theshaderoom.com/young-thug-says-he-wo...,2015-02-04,young thug recently exclusive interview gq usu...,,young thug,0.748121,POSITIVE,0.992790,NEGATIVE
87,https://time.com/6192371/young-thug-rap-lyrics...,NaN,,young thug concerns rap lyrics evidence court ...,young thug,0.925110,NEGATIVE,0.748121,POSITIVE
88,https://time.com/6999813/young-thug-trial-expl...,NaN,,heres young thugs trial stands time,young thug,0.868974,NEGATIVE,0.748121,POSITIVE


### TMZ

In [25]:
tmz = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/tmz.csv', index_col=0)
tmz = SentimentAnalyzer(tmz, ['headlines', 'article'])
tmz.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/tmz.csv')
tmz

Processing headlines in batches:   0%|          | 0/15 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/15 [00:00<?, ?it/s]

,publication_date,headlines,article,links,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
1,2024-07-15,know highprofile young thug rico case,young thug photo prince williamswireimage high...,https://people.com/young-thug-ysl-rico-trail-w...,"young thug, gunna, ysl, ysl rico, rico, jeffre...",0.944002,NEGATIVE,0.987650,NEGATIVE
2,2024-07-15,know highprofile young thug rico case,highprofile trial involving rapper young thug ...,https://people.com/young-thug-ysl-rico-trail-w...,"young thug, gunna, ysl, ysl rico, rico, jeffre...",0.944002,NEGATIVE,0.986918,NEGATIVE
3,2024-11-01,young thugs girlfriend mariah scientist relati...,young thug speaks onstage revolt summit novemb...,https://people.com/who-is-mariah-the-scientist...,"young thug, rico, sb, williams",0.861573,NEGATIVE,0.988755,NEGATIVE
4,2024-11-01,young thugs girlfriend mariah scientist relati...,young thug special duet partner life girlfrien...,https://people.com/who-is-mariah-the-scientist...,"young thug, rico, sb, williams",0.861573,NEGATIVE,0.993752,NEGATIVE
5,2024-10-31,young thug pleads guilty criminal activity cha...,young thug photo paras griffingetty young thug...,https://people.com/young-thug-pleads-guilty-in...,"young thug, gunna, ysl, rico, sb, williams, ro...",0.935545,NEGATIVE,0.991841,NEGATIVE
...,...,...,...,...,...,...,...,...,...
236,2017-07-13,new england patriots danny amendola confirms h...,people probably familiar danny amendola reason...,https://people.com/style/danny-amendola-signs-...,young thug,0.995704,POSITIVE,0.779582,POSITIVE
237,2022-01-06,snl sets ariana debose host roddy ricch musica...,ariana debose roddy ricch photo getty saturday...,https://people.com/tv/snl-sets-first-2022-show...,young thug,0.987183,POSITIVE,0.986662,NEGATIVE
238,2022-01-06,snl sets ariana debose host roddy ricch musica...,saturday night live thursday nbc sketch comedy...,https://people.com/tv/snl-sets-first-2022-show...,young thug,0.987183,POSITIVE,0.986561,NEGATIVE
239,2020-10-27,bet hip hop awards know starstudded event,roddy ricch megan thee stallion drake photo ge...,https://people.com/music/bet-hip-hop-awards-20...,"young thug, gunna",0.965520,POSITIVE,0.844183,NEGATIVE


### XXL

In [26]:
xxl = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Magazine/xxl.csv', index_col=0)
xxl = SentimentAnalyzer(xxl, ['headlines', 'article'])
xxl.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Magazine/xxl.csv')
xxl

Processing headlines in batches:   0%|          | 0/49 [00:00<?, ?it/s]

Processing article in batches:   0%|          | 0/49 [00:00<?, ?it/s]

,publication_date,headlines,snippet,article,links,Keywords,score_headlines,Sentiment_headlines,score_article,Sentiment_article
1,2024-12-04,jurors speak young thug trial time,jurors young thug ysl rico trial finally speak...,jurors young thug ysl rico trial finally speak...,https://www.xxlmag.com/young-thug-jurors-speak...,"young thug, ysl, ysl rico, rico, yak gotti, ya...",0.969759,NEGATIVE,0.990621,NEGATIVE
2,2024-11-12,young thug hits studio lil baby future travis ...,read hiphop reacts young thug released jail wi...,cop xxl merch nowyoung thug studio cooking fut...,https://www.xxlmag.com/young-thug-studio-lil-b...,"young thug, ysl, ysl rico, rico, brian steel",0.998152,POSITIVE,0.992654,NEGATIVE
3,2024-11-11,young thug comments gunna,young thug tweet puzzles fans current status y...,cop xxl merch nowyoung thugs recent tweet gunn...,https://www.xxlmag.com/young-thug-comments-gun...,"young thug, gunna, ysl, ysl rico, rico",0.963587,NEGATIVE,0.997170,NEGATIVE
4,2024-11-13,confused going young thug gunna,curious case status young thug gunna hot topic...,cop xxl merch nowthe curious case status young...,https://www.xxlmag.com/young-thug-gunna-relati...,"young thug, gunna, ysl, ysl rico, rico, williams",0.998110,NEGATIVE,0.997066,NEGATIVE
5,2024-11-13,joe budden thinks young thug fool turn,read confused going young thug gunna status yo...,cop xxl merch nowjoe budden insists foolish yo...,https://www.xxlmag.com/joe-budden-young-thug-f...,"young thug, gunna, ysl, ysl rico, rico",0.997719,NEGATIVE,0.995547,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...
1316,2022-02-18,break presents saucy santana,case saucy santana originally young makeup art...,people stardom excel case saucy santana origin...,https://www.xxlmag.com/saucy-santana-interview...,"gunna, kendrick",0.999206,POSITIVE,0.699261,NEGATIVE
1317,2022-02-21,heres great song great hiphop albums recent me...,coin somber hopeful sounds rod wave consistent...,nas stated hiphop dead years ago genre continu...,https://www.xxlmag.com/one-great-song-great-hi...,"young thug, gunna",0.999815,POSITIVE,0.974428,POSITIVE
1321,2020-11-20,young thug confirms paying lil baby focus rap ...,chatting young thug tip asked head young stone...,lil baby humbly attributed start rap game youn...,https://www.xxlmag.com/young-thug-paid-lil-bab...,young thug,0.917053,NEGATIVE,0.994374,NEGATIVE
1323,2020-11-27,young thug says paid attention andre outkast,think eclectic styles young thug andr crossed ...,think eclectic styles young thug andr crossed ...,https://www.xxlmag.com/young-thug-andre-3000-n...,young thug,0.534543,NEGATIVE,0.983688,NEGATIVE


## Internet

### KingSlime

In [30]:
kingslime = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Internet/KingSlime.csv', index_col=0)
kingslime = SentimentAnalyzer(kingslime, ['description'])
os.makedirs('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/', exist_ok=True)
kingslime.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/KingSlime.csv')
kingslime

Processing description in batches:   0%|          | 0/1 [00:00<?, ?it/s]

,id,name,description,duration_ms,release_date,Keywords,score_description,Sentiment_description
0,12KwTV6DjxnAbK8g4xmAOz,Introducing: King Slime,omnystudiocomlistener privacy information,185103,2023-07-17,NaN,0.984124,NEGATIVE
1,0CXambVuLpaiFDFnMfpyfQ,Episode 1: Stoner or Slime,past decade rapper young thug international su...,2857691,2023-08-15,"young thug, ysl, jeffrey, williams",0.877848,NEGATIVE
2,0WCi9RYzY6TIKIcniOQbbO,Episode 2: Bleveland Avenue,young thug raised south atlantas notorious str...,2963983,2023-08-22,young thug,0.595574,POSITIVE
3,0IKl0c79gaLRANmU6nqqaj,Episode 3: Take It To Trial Pt. 1,haunting scream echoes courtroom ysl trial cau...,3476741,2023-08-29,"ysl, rico",0.711504,POSITIVE
4,77KzOuARZVqbXCQrWeNThj,Episode 4: Take It To Trial Pt. 2,prosecutors claim ysl dangerous criminal organ...,2112130,2023-09-05,"ysl, rico, lil rod",0.985058,NEGATIVE
5,4MvSFXwR9N3cpmH1b6mRBQ,Episode 5: Drop Bars,prosecutors ysl trial cite lyrics dozen songs ...,3312901,2023-09-12,"young thug, ysl",0.959355,NEGATIVE
6,4hcXbJTtqi0FTcLc8JT5ht,Episode 6: I Came From Nothing,young thugs rise stardom current legal entangl...,4278648,2023-09-20,"young thug, ysl",0.819681,POSITIVE
7,7hzN9TkV3vxnschSmeGv0m,Episode 7: Back At It,december ysl defendant sergio kitchens rapper ...,2815582,2023-12-05,"gunna, ysl, sergio kitchens, sergio, kitchens",0.932961,NEGATIVE
8,4OaikUk55rt4GStDqhCkO7,Episode 8: F#@& Rice Street,time opening statements began ysl trial remain...,3140257,2023-12-13,ysl,0.986446,NEGATIVE
9,2APotk34K5UqN75B8AaNtj,"Court Report 1: LeBron James, Serena Williams ...",jury seated ysl trial underway king slime team...,2281482,2024-01-24,ysl,0.933903,NEGATIVE


### LawAndOrder YouTube

In [32]:
lawandorder = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Internet/LawAndOrder_YouTube.csv', index_col=0)
lawandorder = SentimentAnalyzer(lawandorder, ['comment'])
lawandorder.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/LawAndOrder_YouTube.csv')
lawandorder

Processing comment in batches:   0%|          | 0/1544 [00:00<?, ?it/s]

,video_id,title,views,video likes,total comment,upload_date,comment,comment likes,replies,published_at,Keywords,score_comment,Sentiment_comment
0,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,face definition hate facewithsteamfromnoseface...,0.0,0.0,2024-12-13 03:20:50+00:00,"young thug, ysl",0.999104,NEGATIVE
1,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,time watching beginning,2.0,0.0,2024-12-07 02:14:54+00:00,"young thug, ysl",0.988087,POSITIVE
2,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,love lying b hope jury pay attention,1.0,0.0,2024-11-27 05:13:45+00:00,"young thug, ysl",0.995111,POSITIVE
3,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,time watching opening da love sneaky day wow,2.0,0.0,2024-11-10 23:41:25+00:00,"young thug, ysl",0.999434,POSITIVE
4,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,strength wolf sit love playing defendants witn...,1.0,0.0,2024-11-09 23:39:06+00:00,"young thug, ysl",0.997922,POSITIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24697,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,judge let speak let prosecutors cut defense de...,203.0,10.0,2024-06-20 21:06:50+00:00,"young thug, rico",0.989538,NEGATIVE
24698,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,ya ai nt,3.0,0.0,2024-06-20 21:06:30+00:00,"young thug, rico",0.580307,POSITIVE
24699,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,judge disgrace,338.0,6.0,2024-06-20 21:05:03+00:00,"young thug, rico",0.998399,NEGATIVE
24700,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,st,3.0,0.0,2024-06-20 21:02:17+00:00,"young thug, rico",0.972638,POSITIVE


### Youtube Artist

In [37]:
youtube_artist = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Internet/YouTube Artist.csv', index_col=0)
youtube_artist = SentimentAnalyzer(youtube_artist, ['comment'], batch_size = 120)
youtube_artist.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/YouTube Artist.csv')
youtube_artist

Processing comment in batches:   0%|          | 0/1589 [00:00<?, ?it/s]

,video_id,channel_title,video_duration,upload_date,title,description,views,video likes,total comment,published at,comment,comment likes,Keywords,score_comment,Sentiment_comment
0,cdsQpgHq4rs,Young Thug,0 days 00:00:16,2021-10-14 17:52:19+00:00,Drop 💕 if you’re ready for PUNK ⬇️,"💕 PUNK TONIGHT 💕\nStream ""Tick Tock"" on all pl...",322124.0,14682,842.0,2024-12-26 14:22:57+00:00,heartsuit nigga thug prison think,0.0,NaN,0.988526,NEGATIVE
1,cdsQpgHq4rs,Young Thug,0 days 00:00:16,2021-10-14 17:52:19+00:00,Drop 💕 if you’re ready for PUNK ⬇️,"💕 PUNK TONIGHT 💕\nStream ""Tick Tock"" on all pl...",322124.0,14682,842.0,2024-12-26 12:52:28+00:00,like sahbabii,0.0,NaN,0.928133,POSITIVE
2,cdsQpgHq4rs,Young Thug,0 days 00:00:16,2021-10-14 17:52:19+00:00,Drop 💕 if you’re ready for PUNK ⬇️,"💕 PUNK TONIGHT 💕\nStream ""Tick Tock"" on all pl...",322124.0,14682,842.0,2024-12-22 20:08:15+00:00,price gee,0.0,NaN,0.958742,NEGATIVE
3,cdsQpgHq4rs,Young Thug,0 days 00:00:16,2021-10-14 17:52:19+00:00,Drop 💕 if you’re ready for PUNK ⬇️,"💕 PUNK TONIGHT 💕\nStream ""Tick Tock"" on all pl...",322124.0,14682,842.0,2024-12-11 02:44:03+00:00,trey love ya birthday,0.0,NaN,0.999756,POSITIVE
4,cdsQpgHq4rs,Young Thug,0 days 00:00:16,2021-10-14 17:52:19+00:00,Drop 💕 if you’re ready for PUNK ⬇️,"💕 PUNK TONIGHT 💕\nStream ""Tick Tock"" on all pl...",322124.0,14682,842.0,2024-12-06 12:37:53+00:00,love yt xx greenheartwhiteheartlightblueheartp...,0.0,NaN,0.983285,POSITIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15083,jjXqLHIwQVs,YFN Lucci,0 days 00:02:26,2020-12-04 05:00:17+00:00,YFN Lucci - Wish Me Well 3 Coming (Official Au...,"The official audio for ""Wish Me Well 3 Coming""...",665968.0,7394,125.0,2020-12-04 05:06:42+00:00,yessir,0.0,NaN,0.996651,POSITIVE
15084,jjXqLHIwQVs,YFN Lucci,0 days 00:02:26,2020-12-04 05:00:17+00:00,YFN Lucci - Wish Me Well 3 Coming (Official Au...,"The official audio for ""Wish Me Well 3 Coming""...",665968.0,7394,125.0,2020-12-04 05:06:41+00:00,,0.0,NaN,0.748121,POSITIVE
15085,jjXqLHIwQVs,YFN Lucci,0 days 00:02:26,2020-12-04 05:00:17+00:00,YFN Lucci - Wish Me Well 3 Coming (Official Au...,"The official audio for ""Wish Me Well 3 Coming""...",665968.0,7394,125.0,2020-12-04 05:03:47+00:00,,1.0,NaN,0.748121,POSITIVE
15086,jjXqLHIwQVs,YFN Lucci,0 days 00:02:26,2020-12-04 05:00:17+00:00,YFN Lucci - Wish Me Well 3 Coming (Official Au...,"The official audio for ""Wish Me Well 3 Coming""...",665968.0,7394,125.0,2020-12-04 05:03:35+00:00,shi,20.0,NaN,0.928585,POSITIVE


In [38]:
youtube_artist['Sentiment_comment'].value_counts()

,count
Sentiment_comment,
POSITIVE,99344
NEGATIVE,91301


### Reddit

In [40]:
reddit = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Internet/reddit.csv', index_col=0)
reddit = SentimentAnalyzer(reddit, ['title', 'text'])
reddit.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/Reddit.csv')
reddit

Processing title in batches:   0%|          | 0/130 [00:00<?, ?it/s]

Processing text in batches:   0%|          | 0/130 [00:00<?, ?it/s]

,Published Date,title,text,comments,subreddit,search term,url,Keywords,score_title,Sentiment_title,score_text,Sentiment_text
0,2073-12-04 03:51:45,young thug episode new hbo show chillin island...,,55.0,YoungThug,young thug,https://www.reddit.com/r/YoungThug/comments/ri...,young thug,0.979756,NEGATIVE,0.748121,POSITIVE
4,2075-12-22 10:23:24,savage called young thug birthday girl,,7.0,YoungThug,young thug,https://www.reddit.com/r/YoungThug/comments/zw...,young thug,0.978858,NEGATIVE,0.748121,POSITIVE
5,2069-12-09 00:29:37,young thugs fun custom nintendo ds games regul...,,9.0,YoungThug,young thug,https://www.reddit.com/r/YoungThug/comments/ed...,young thug,0.877213,POSITIVE,0.748121,POSITIVE
6,2079-02-28 17:50:11,young thug amp team learns secret meeting took...,,83.0,YoungThug,young thug,https://www.reddit.com/r/YoungThug/comments/1e...,young thug,0.842308,NEGATIVE,0.748121,POSITIVE
7,2078-06-21 19:58:45,young thug underrated,,93.0,YoungThug,young thug,https://www.reddit.com/r/YoungThug/comments/1b...,young thug,0.922919,NEGATIVE,0.748121,POSITIVE
...,...,...,...,...,...,...,...,...,...,...,...,...
931,2076-02-08 23:41:18,registered yfn lucci dirty diana,,3.0,Atlantology,yfn lucci,https://www.reddit.com/r/Atlantology/comments/...,"yfn lucci, yfn",0.991531,NEGATIVE,0.748121,POSITIVE
932,2074-11-29 01:56:35,lil durk rich forever feat yfn lucci,,2.0,Atlantology,yfn lucci,https://www.reddit.com/r/Atlantology/comments/...,"yfn lucci, yfn",0.998722,POSITIVE,0.748121,POSITIVE
934,2079-05-21 20:50:39,know song,ig live king von listening pretty sure yfn luc...,11.0,Atlantology,yfn lucci,https://www.reddit.com/r/Atlantology/comments/...,"yfn lucci, yfn",0.995233,POSITIVE,0.917803,NEGATIVE
935,2078-11-20 00:12:36,bodycam yfn lucci stopped cops car impounded,,8.0,Atlantology,yfn lucci,https://www.reddit.com/r/Atlantology/comments/...,"yfn lucci, yfn",0.992647,NEGATIVE,0.748121,POSITIVE


### YSL Hashtag

In [45]:
ysl = pd.read_csv('/content/drive/MyDrive/Young_Thug/Data/Cleaned Data/Topic Modeling/Internet/ysl.csv', index_col = 0)
ysl = SentimentAnalyzer(ysl, ['Tweets'])
ysl.to_csv('/content/drive/MyDrive/Young_Thug/Sentiment Analysis/Internet/ysl.csv')
ysl

Processing Tweets in batches:   0%|          | 0/47 [00:00<?, ?it/s]

,Timestamp,Name,Handle,Verified,Tweets,Comments,Retweets,Likes,Analytics,Tags,Mentions,Keywords,score_Tweets,Sentiment_Tweets
0,2024-12-27 03:50:39+00:00,KZ,@kz3354,False,ysltrial watchers shell kels interview dropped...,0,0,0,1,['#ysltrial'],[],ysl,0.995269,NEGATIVE
1,2024-12-27 03:34:07+00:00,chrissy paradis,@chrissyparadis,True,ada love nathan wade judge glanville fani will...,0,0,1,64,"['#ysltrial', '#ysl']",[],"ysl, glanville",0.997023,NEGATIVE
2,2024-12-27 00:43:45+00:00,Valerie,@queen__valerie_,True,longest trial ga history longest series netfli...,1,0,5,228,"['#ysltrial', '#netflix']",[],ysl,0.950680,NEGATIVE
3,2024-12-27 00:11:17+00:00,Valerie,@queen__valerie_,True,agree like watching broad trailer witnessed en...,3,0,3,518,['#ysltrial'],[],ysl,0.880154,NEGATIVE
4,2024-12-26 23:45:13+00:00,Sunglasses x Sneakers,@SceneByAshlix,False,judge whitaker anytime attorney love opened mo...,0,0,1,33,['#YSLTrial'],[],"ysl, whitaker",0.967355,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,2022-12-22 04:48:27+00:00,King Jaffe Joseph,@King_JaffeJoe,False,latest ysl trial young thugs brother joins gun...,0,1,2,1600,"['#torylaneztrial', '#youngthugtrial', '#ysltr...",['@YouTube'],"young thug, gunna, ysl",0.992779,NEGATIVE
744,2022-12-21 21:55:44+00:00,tee dee,@Agnb_223Twon,False,judge phillip banks ai nt taking easy ysl youn...,0,0,1,134,"['#youngthug', '#ysl', '#ysltrial']",[],ysl,0.965056,NEGATIVE
745,2022-12-19 17:01:40+00:00,Ray Ray Wick,@KingBigWuan,False,home smoking fat blunt bro probably watching t...,0,0,0,78,['#ysltrial'],['@1GunnaGunna'],ysl,0.998466,NEGATIVE
746,2022-12-22 15:55:51+00:00,YSL TRIAL TRACKER,@YSLTRIALTRACKER,False,entries far nt miss chance owning true piece h...,0,2,4,2000,"['#ysltrial', '#freeyoungthug']",['@9PM'],ysl,0.990348,NEGATIVE
